In [ ]:
!pip install "protobuf<=3.20.3"

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import os
import cv2
import random
import numpy as np
import shutil
import math
import seaborn as sns
import pandas as pd
from tqdm import tqdm
import time
import os

import tensorflow as tf
from tensorflow.keras.utils import img_to_array
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array

from keras.optimizers import Adam, RMSprop
from keras.layers import Input, Concatenate, ZeroPadding2D, BatchNormalization
from keras.layers import Dense, Dropout, Activation
from keras.layers import Conv2D, MaxPooling2D, AveragePooling2D, GlobalAveragePooling2D
from keras.layers import BatchNormalization, ZeroPadding2D, Concatenate, Input
from keras.models import Model
from keras.preprocessing import image
from keras.callbacks import ModelCheckpoint
import keras.backend as K
import keras

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split

print("TF version:", tf.version)
print("Keras version:", tf.keras.version)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.utils import img_to_array
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array

from keras.optimizers import Adam, RMSprop
from keras.layers import Input, Concatenate, ZeroPadding2D, BatchNormalization
from keras.layers import Dense, Dropout, Activation
from keras.layers import Conv2D, MaxPooling2D, AveragePooling2D, GlobalAveragePooling2D
from keras.layers import BatchNormalization, ZeroPadding2D, Concatenate, Input
from keras.models import Model
from keras.preprocessing import image
from keras.callbacks import ModelCheckpoint
import keras.backend as K


from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss

print("TF version:", tf.version)
print("Keras version:", tf.keras.version)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

In [ ]:
def densenet121_model(
    img_rows,
    img_cols,
    color_type=1,
    nb_dense_block=4,
    growth_rate=32,
    nb_filter=64,
    reduction=0.5,
    dropout_rate=0.0,
    weight_decay=1e-4,
    num_classes=None,
):
    """
    DenseNet 121 Model for Keras

    Model Schema is based on
    https://github.com/flyyufelix/DenseNet-Keras

    # Returns
        A Keras model instance.
    """

    # Handle Dimension Ordering for different backends
    # global concat_axis
    img_input = Input(shape=(img_rows, img_cols, color_type), name="data")
    # concat_axis = 3

    # From architecture for ImageNet (Table 1 in the paper)
    nb_filter = 64
    nb_layers = [6, 12, 24, 16]  # For DenseNet-121

    # Initial convolution
    # x = Conv2D(nb_filter, 7, 7, subsample=(2, 2), name='conv1', bias=False)(img_input)
    x = Conv2D(
        filters=nb_filter,
        kernel_size=(7, 7),
        strides=(2, 2),
        use_bias=False,
        name="conv1",
    )(img_input)

    x = BatchNormalization(axis=-1)(x)
    # x = Scale(axis=-1)(x)
    x = Activation("relu")(x)

    x = MaxPooling2D((3, 3), strides=(2, 2))(x)

    # Add dense blocks
    for block_idx in range(nb_dense_block - 1):
        stage = block_idx + 2
        x, nb_filter = dense_block(
            x,
            stage,
            nb_layers[block_idx],
            nb_filter,
            growth_rate,
            dropout_rate=dropout_rate,
        )

        # Add transition_block
        x = transition_block(x, stage, nb_filter, dropout_rate=dropout_rate)
        nb_filter = int(nb_filter)

    final_stage = stage + 1
    x, nb_filter = dense_block(
        x, final_stage, nb_layers[-1], nb_filter, growth_rate, dropout_rate=dropout_rate
    )

    x = BatchNormalization(axis=-1)(x)
    # x = Scale(axis=-1 )(x)
    x = Activation("relu")(x)

    # x_fc = GlobalAveragePooling2D()(x)
    # x_fc = Dense(1000)(x_fc)
    # x_fc = Activation('softmax')(x_fc)
    # model = Model(img_input, x_fc)

    # The method below works since pre-trained weights are stored in layers but not in the model
    x_newfc = GlobalAveragePooling2D()(x)
    x_newfc = Dense(num_classes)(x_newfc)
    x_newfc = Activation("softmax")(x_newfc)

    model = Model(img_input, x_newfc)

    # ADAM OPTIMIZER
    # adm = Adam(learning_rate=1e-4)
    # model.compile(optimizer=adm, loss='categorical_crossentropy', metrics=['accuracy'])

    # RMSprop OPTIMIZER
    RMSp = RMSprop(learning_rate=1e-4)
    model.compile(optimizer=RMSp, loss="categorical_crossentropy", metrics=["accuracy"])

    return model

In [ ]:
def conv_block(x, stage, branch, nb_filter, dropout_rate=None):
    """Apply BatchNorm, Relu, bottleneck 1x1 Conv2D, 3x3 Conv2D, and option dropout"""

    # 1x1 Convolution (Bottleneck layer)
    inter_channel = nb_filter * 4
    x = BatchNormalization(axis=-1)(x)
    # x = Scale(axis=-1)(x)
    x = Activation("relu")(x)
    x = Conv2D(inter_channel, (1, 1), use_bias=False)(x)  # argumen lama diganti
    # x = Conv2D(inter_channel, 1, 1, bias=False)(x)

    if dropout_rate:
        x = Dropout(dropout_rate)(x)

    # 3x3 Convolution
    x = BatchNormalization(axis=-1)(x)
    # x = Scale(axis=-1)(x)
    x = Activation("relu")(x)
    x = ZeroPadding2D((1, 1))(x)
    x = Conv2D(nb_filter, (3, 3), use_bias=False)(x)  # ganti format argumen
    # x = Conv2D(nb_filter, 3, 3, bias=False)(x)

    if dropout_rate:
        x = Dropout(dropout_rate)(x)

    return x


def transition_block(x, stage, nb_filter, dropout_rate=None):
    """Apply BatchNorm, 1x1 Convolution, averagePooling, optional compression, dropout"""

    x = BatchNormalization(axis=-1)(x)
    # x = Scale(axis=-1)(x)
    x = Activation("relu")(x)
    x = Conv2D(int(nb_filter), (1, 1), use_bias=False)(x)  # format baru
    # x = Conv2D(int(nb_filter), 1, 1, bias=False)(x)

    if dropout_rate:
        x = Dropout(dropout_rate)(x)

    x = AveragePooling2D((2, 2), strides=(2, 2))(x)

    return x


def dense_block(
    x, stage, nb_layers, nb_filter, growth_rate, dropout_rate=None, grow_nb_filters=True
):
    """Build a dense_block where the output of each conv_block is fed to subsequent ones
    # Arguments
        x: input tensor
        stage: index for dense block
        nb_layers: the number of layers of conv_block to append to the model.
        nb_filter: number of filters
        growth_rate: growth rate
        grow_nb_filters: flag to decide to allow number of filters to grow
    """

    concat_feat = x

    for i in range(nb_layers):
        branch = i + 1
        x = conv_block(concat_feat, stage, branch, growth_rate, dropout_rate)
        concat_feat = Concatenate(axis=-1)([concat_feat, x])

        if grow_nb_filters:
            nb_filter += growth_rate

    return concat_feat, nb_filter

In [ ]:
data_path = (
    "/kaggle/input/datasets/dewamardana/dataset-manual-selection/dataset_centralcrop"
)

images = []
labels = []

for subfolder in os.listdir(data_path):

    subfolder_path = os.path.join(data_path, subfolder)
    if not os.path.isdir(subfolder_path):
        continue

    for image_filename in os.listdir(subfolder_path):
        image_path = os.path.join(subfolder_path, image_filename)
        images.append(image_path)

        labels.append(subfolder)

data = pd.DataFrame({"image": images, "label": labels})
data.head()
data.shape

In [ ]:
import matplotlib.pyplot as plt

class_counts = data["label"].value_counts().sort_index()
class_counts

plt.figure(figsize=(12, 6))
bars = plt.bar(class_counts.index, class_counts.values)

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2, height, str(height), ha="center", va="bottom"
    )

plt.xlabel("Kelas")
plt.ylabel("Jumlah Data")
plt.title("Distribusi Jumlah Data pada Setiap Kelas")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
strat = data["label"]
train_df, dummy_df = train_test_split(
    data, train_size=0.80, shuffle=True, random_state=123, stratify=strat
)

strat = dummy_df["label"]
valid_df, test_df = train_test_split(
    dummy_df, train_size=0.5, shuffle=True, random_state=123, stratify=strat
)

print("Training set shape:", train_df.shape)
print("Validation set shape:", valid_df.shape)
print("Test set shape:", test_df.shape)

In [ ]:
batch_size = 16
img_size = (256, 256)
channels = 3
img_shape = (img_size[0], img_size[1], channels)


tr_gen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest",
)

ts_gen = ImageDataGenerator()


train_gen = tr_gen.flow_from_dataframe(
    train_df,
    x_col="image",
    y_col="label",
    target_size=img_size,
    class_mode="categorical",
    color_mode="rgb",
    shuffle=True,
    batch_size=batch_size,
)


valid_gen = ts_gen.flow_from_dataframe(
    valid_df,
    x_col="image",
    y_col="label",
    target_size=img_size,
    class_mode="categorical",
    color_mode="rgb",
    shuffle=False,
    batch_size=batch_size,
)

test_gen = ts_gen.flow_from_dataframe(
    test_df,
    x_col="image",
    y_col="label",
    target_size=img_size,
    class_mode="categorical",
    color_mode="rgb",
    shuffle=False,
    batch_size=batch_size,
)

In [ ]:
from keras.callbacks import ModelCheckpoint

if __name__ == "__main__":

    # Example to fine-tune on 3000 samples from Cifar10

    img_rows, img_cols = 256, 256  # Resolution of inputs
    channel = 3
    num_classes = 12
    nb_epoch = 16

    # Load Cifar10 data. Please implement your own load_data() module for your own dataset
    # X_train, Y_train, X_valid, Y_valid = load_data()

    # Load our model    print("Devices:", tf.config.list_physical_devices())

    model = densenet121_model(
        img_rows=img_rows,
        img_cols=img_cols,
        color_type=channel,
        num_classes=num_classes,
    )
    filepath = "bestmodel.keras"
    checkpoint = ModelCheckpoint(
        filepath, monitor="val_accuracy", verbose=1, save_best_only=True, mode="max"
    )
    callbacks_list = [checkpoint]
    # Start CNN

    # =====================================================
    # TRAINING START TIME
    # =====================================================
    training_start_time = time.time()

    history = model.fit(
        train_gen,
        epochs=nb_epoch,
        shuffle=False,
        verbose=1,
        validation_data=valid_gen,
        callbacks=callbacks_list,
    )
    # =====================================================
    # TRAINING END TIME
    # =====================================================
    training_end_time = time.time()

    # =====================================================
    # TOTAL TRAINING TIME
    # =====================================================
    total_training_time = training_end_time - training_start_time

    print(f"\nTotal Training Time: {total_training_time:.2f} seconds")

    # =====================================================
    # MODEL SIZE
    # =====================================================
    model_size_mb = os.path.getsize("bestmodel.keras") / (1024 * 1024)

    print(f"Model Size: {model_size_mb:.2f} MB")

In [ ]:
# =============================
# 1. Evaluasi Train & Validation (dari history)
# =============================
train_acc = history.history["accuracy"][-1]
train_loss = history.history["loss"][-1]
val_acc = history.history["val_accuracy"][-1]
val_loss = history.history["val_loss"][-1]

print("=== TRAINING METRICS ===")
print("Training Accuracy :", train_acc)
print("Training Loss     :", train_loss)
print("\n=== VALIDATION METRICS ===")
print("Validation Accuracy :", val_acc)
print("Validation Loss     :", val_loss)

In [ ]:
# =============================
# 2. Plot Training vs Validation Accuracy & Loss
# =============================
import matplotlib.pyplot as plt

epochs = range(len(history.history["accuracy"]))

plt.figure(figsize=(10, 5))
plt.plot(epochs, history.history["accuracy"], label="Training Accuracy")
plt.plot(epochs, history.history["val_accuracy"], label="Validation Accuracy")
plt.legend()
plt.grid(True)
plt.title("Training vs Validation Accuracy")
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(epochs, history.history["loss"], label="Training Loss")
plt.plot(epochs, history.history["val_loss"], label="Validation Loss")
plt.legend()
plt.grid(True)
plt.title("Training vs Validation Loss")
plt.show()

In [ ]:
from tensorflow.keras.utils import to_categorical

# =============================
# 3. Prediksi Validation Set
# =============================
ypred = model.predict(valid_gen, verbose=1)

# ground truth
ytrue = valid_gen.classes
ytrue_cat = to_categorical(ytrue, num_classes=num_classes)

In [ ]:
# =============================
# 4. Evaluasi Test Dataset
# =============================
test_loss, test_acc = model.evaluate(test_gen, verbose=1)

print("\n=== TEST METRICS ===")
print("Test Accuracy :", test_acc)
print("Test Loss     :", test_loss)

In [ ]:
# =============================
# 5. Akurasi Manual
# =============================
predicted_labels = np.argmax(ypred, axis=1)
accurate = np.sum(predicted_labels == ytrue)
total = len(ytrue)

print("\n=== MANUAL ACCURACY CHECK ===")
print("Total Data     :", total)
print("Correct Predict:", accurate)
print("Wrong Predict  :", total - accurate)
print("Accuracy (%)   :", accurate / total * 100)

In [ ]:
# =============================
# 6. Log Loss
# =============================
from sklearn.metrics import log_loss

val_logloss = log_loss(ytrue_cat, ypred)
print("\nValidation Log Loss:", val_logloss)

In [ ]:
# =====================================================
# CLASSIFICATION START TIME
# =====================================================
classification_start_time = time.time()

# =====================================================
# PREDIKSI TEST SET
# =====================================================
test_predictions = model.predict(test_gen, verbose=1)

# =====================================================
# CLASSIFICATION END TIME
# =====================================================
classification_end_time = time.time()

# =====================================================
# TOTAL CLASSIFICATION TIME
# =====================================================
total_classification_time = classification_end_time - classification_start_time

print(f"Total Classification Time: " f"{total_classification_time:.2f} seconds")

In [ ]:
# =====================================================
# PREDICTED LABEL
# =====================================================
predicted_labels = np.argmax(test_predictions, axis=1)

# =====================================================
# TRUE LABEL
# =====================================================
true_labels = test_gen.classes

# =====================================================
# ACCURACY
# =====================================================
accuracy = accuracy_score(true_labels, predicted_labels)

# =====================================================
# PRECISION
# =====================================================
precision = precision_score(true_labels, predicted_labels, average="weighted")

# =====================================================
# RECALL
# =====================================================
recall = recall_score(true_labels, predicted_labels, average="weighted")

# =====================================================
# F1 SCORE
# =====================================================
f1 = f1_score(true_labels, predicted_labels, average="weighted")

In [ ]:
# =====================================================
# FINAL RESULT
# =====================================================
print("\n===================================")
print("FINAL TEST RESULT - SKEMA 1")
print("===================================")

print("================ Uji Performa ==============")
print(f"Accuracy               : {accuracy:.4f}")
print(f"Precision              : {precision:.4f}")
print(f"Recall                 : {recall:.4f}")
print(f"F1-Score               : {f1:.4f}")
print("================ Uji Komputasi ==============")
print(f"Training Time (s)      : {total_training_time:.2f}")
print(f"Classification Time(s) : {total_classification_time:.2f}")
print(f"Model Size (MB)        : {model_size_mb:.2f}")

In [ ]:
# =============================
# 7. Confusion Matrix & Classification Report
# =============================
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

cm = confusion_matrix(true_labels, predicted_labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

print("\n=== Classification Report ===")
print(
    classification_report(
        true_labels, predicted_labels, target_names=list(train_gen.class_indices.keys())
    )
)